In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
query = """
-- agrupo por order id y tomo el valor total de la orden (aparecen varios pagos de la misma order id porque se compro mas de un mismo producto u otro producto)
WITH payments_agg AS (
    SELECT 
        order_id,
        SUM(payment_value) AS total_payment_value
    FROM `catalog_brazilian-e-commerce`.silver.olist_order_payments
    GROUP BY order_id
)

SELECT
    o.order_id,
    oi.order_item_id,
    o.customer_id,
    oi.product_id,
    oi.seller_id,

    -- métricas -> total_item_value es el valor de 1 item o sea de 1 vez que aparecio el order_id de tantas que aparece, mientras que total payment value es la suma de todas las veces que aparecio el order id
    oi.price,
    oi.freight_value,
    (oi.price + oi.freight_value) AS total_item_value,
    p.total_payment_value,

    -- fechas
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    -- estado
    o.order_status

FROM `catalog_brazilian-e-commerce`.silver.olist_orders o
JOIN `catalog_brazilian-e-commerce`.silver.olist_order_items oi
    ON o.order_id = oi.order_id
LEFT JOIN payments_agg p
    ON o.order_id = p.order_id
"""

df = spark.sql(query)

In [0]:
df.limit(10).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`catalog_brazilian-e-commerce`.gold.fact_sales")

In [0]:
%sql
select * from `catalog_brazilian-e-commerce`.gold.fact_sales
limit 10